In [1]:
import pandas as pd
import psycopg2
from utils import train_utils
import importlib
importlib.reload(train_utils)
from utils.train_utils import *

# Define your connection parameters
conn = psycopg2.connect(
    host="localhost",
    database="smartinvestor_ln",
    user="postgres",
    password="postgres"
)

c:\Users\HANJ29\Development\vdev1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import yaml
import os

labels_dict = {
    "STD": "top_or_bottom",
    "STAT": "top_or_bottom_stat",
    "VOL": "top_bottom_volatility_stat",
    "STDOPT": "top_or_bottom_optimized",
    "STATOPT": "top_or_bottom_stat_optimized",
    "VOLOPT": "top_bottom_volatility_optimized",
}

labels = [
    *labels_dict.values()
]

# Define mapping for ts_code leading chars
ts_code_map = {
    "60": "SH",    # Shanghai A-shares
    "3": "CYB",    # ChiNext
    "688": "KCB",  # STAR Market
    "0": "SZ",     # Shenzhen A-shares
    "8": "BJ",     # Beijing Stock Exchange
    "9": "BJ"      # Beijing Stock Exchange (also used for BJ codes)
}

def read_features_from_yaml(yaml_path, feature_type=None):
    """
    Reads a list of features from a YAML file.

    Args:
        yaml_path (str): Path to the YAML file.

    Returns:
        list: List of features.
    """
    with open(yaml_path, "r") as file:
        data = yaml.safe_load(file)
    # Assumes the YAML file contains a top-level key 'features'
    return data.get(feature_type, [])


# Assuming the features YAML file is named 'features.yaml'
def get_config_folder(features_yaml_filename="features.yaml"):
    for root, dirs, files in os.walk("."):
        if features_yaml_filename in files:
            return os.path.abspath(root)
    return None

def replace_bt_values(df, columns):
    """
    Replace values in specified columns: 'B' -> 1, 'T' -> 2, others -> 0.

    Args:
        df (pd.DataFrame): The dataframe to modify.
        columns (list): List of column names to process.

    Returns:
        pd.DataFrame: DataFrame with replaced values in specified columns.
    """
    mapping = {'B': 1, 'T': 2}
    df_copy = df.copy()
    for col in columns:
        df_copy[col] = df_copy[col].map(mapping).fillna(0).astype(int)
    return df_copy

def get_df_by_ts_code_prefix(prefix, use_cols=None):
    """
    Returns the DataFrame for the given ts_code prefix by reading the relevant CSV file from ./static/cost/.

    Args:
        prefix (str): The leading ts_code prefix (e.g., '60', '688', '0', '8', '9', '3').

    Returns:
        pd.DataFrame: The filtered DataFrame for the specified market/prefix.
    """
    market = ts_code_map.get(prefix)
    if market is None:
        raise ValueError(f"Prefix '{prefix}' not found in ts_code_map.")
    filename = f"./static/bundle/bundle_dataset_{market}_{prefix}.csv"
    if not os.path.exists(filename):
        raise FileNotFoundError(f"File {filename} does not exist.")
    return pd.read_csv(filename, usecols=use_cols)

In [3]:
csv_surfix = "688"
config_folder = get_config_folder()
feature_tech = read_features_from_yaml(f'{config_folder}/features.yaml', feature_type='feature_tech')
feature_funda = read_features_from_yaml(f'{config_folder}/features.yaml', feature_type='feature_fundamental')
feature_cost = read_features_from_yaml(f'{config_folder}/features.yaml', feature_type='feature_cost')

for feature_list_name in ['feature_tech', 'feature_funda', 'feature_cost']:
    for col in ['freq', "dv_ratio", 'dv_ttm']:
        if col in globals()[feature_list_name]:
            globals()[feature_list_name].remove(col)

In [4]:
market_df = get_df_by_ts_code_prefix(csv_surfix, use_cols=feature_tech + feature_funda + feature_cost + labels)  # Example for Shanghai A-shares

In [5]:
sanity_check_df = market_df[market_df['ts_code'] == '688019.SH']

In [6]:
sanity_check_df = sanity_check_df.sort_values(by='trade_date', ascending=False)
sanity_check_df


,ts_code,trade_date,close_his_low_diff,close_his_high_diff,close_cost_5pct_diff,close_cost_15pct_diff,close_cost_50pct_diff,close_cost_85pct_diff,close_cost_95pct_diff,close_weight_avg_diff,...,ps_200d_25pct_diff,ps_200d_50pct_diff,ps_200d_75pct_diff,ps_200d_90pct_diff,ps_ttm,total_share,float_share,free_share,total_mv,circ_mv
24823,688019.SH,2025-09-30,186.44,-7.06,0.37,0.28,0.09,0.01,-0.01,0.12,...,0.56,0.42,0.30,0.22,17.6692,16855.43,16855.43,11680.0,3850453.47,3850453.47
24824,688019.SH,2025-09-29,190.35,-3.15,0.40,0.31,0.11,0.04,0.01,0.15,...,0.59,0.44,0.33,0.24,17.9716,16855.43,16855.43,11680.0,3916358.18,3916358.18
24825,688019.SH,2025-09-26,181.12,-9.38,0.34,0.27,0.09,0.01,-0.02,0.12,...,0.53,0.39,0.28,0.20,17.2577,16855.43,16855.43,11680.0,3760782.60,3760782.60
24826,688019.SH,2025-09-25,176.34,-11.16,0.32,0.25,0.08,0.00,-0.02,0.11,...,0.49,0.36,0.25,0.18,16.8880,16855.43,16855.43,11680.0,3680213.67,3680213.67
24827,688019.SH,2025-09-24,176.50,-11.00,0.32,0.26,0.10,0.01,-0.02,0.12,...,0.50,0.36,0.25,0.18,16.9003,16855.43,16855.43,11680.0,3682910.54,3682910.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26324,688019.SH,2019-07-26,21.69,-103.11,-0.01,-0.01,-0.05,-0.07,-0.09,-0.05,...,-0.01,-0.04,-0.10,-0.11,35.3353,5310.84,1157.50,1157.5,936035.20,204008.60
26325,688019.SH,2019-07-25,29.60,-95.20,0.13,0.11,0.08,0.04,0.02,0.07,...,0.10,0.06,0.02,0.01,40.2571,5310.84,1157.50,1157.5,1066416.27,232425.12
26326,688019.SH,2019-07-24,23.87,-100.93,0.08,0.05,0.03,-0.01,-0.01,0.03,...,0.01,0.00,-0.03,-0.05,36.6885,5310.84,1157.50,1157.5,971883.35,211821.69
26327,688019.SH,2019-07-23,22.50,-102.30,0.11,0.08,0.01,-0.04,-0.08,0.01,...,-0.02,-0.05,-0.07,-0.08,35.8365,5310.84,1157.50,1157.5,949312.29,206902.34


In [5]:
market_df = replace_bt_values(market_df, labels)

In [8]:
market_df

,ts_code,trade_date,close_his_low_diff,close_his_high_diff,close_cost_5pct_diff,close_cost_15pct_diff,close_cost_50pct_diff,close_cost_85pct_diff,close_cost_95pct_diff,close_weight_avg_diff,...,ps_200d_25pct_diff,ps_200d_50pct_diff,ps_200d_75pct_diff,ps_200d_90pct_diff,ps_ttm,total_share,float_share,free_share,total_mv,circ_mv
0,688001.SH,2025-09-30,13.19,-10.41,0.29,0.16,0.03,-0.03,-0.04,0.05,...,0.27,0.15,0.05,-0.01,7.4040,44537.78,44537.78,8470.00,1406948.61,1406948.61
1,688001.SH,2025-09-29,13.40,-10.20,0.30,0.17,0.05,-0.02,-0.04,0.06,...,0.27,0.15,0.06,0.00,7.4532,44537.78,44537.78,8470.00,1416301.54,1416301.54
2,688001.SH,2025-09-26,13.13,-10.47,0.29,0.17,0.04,-0.03,-0.04,0.05,...,0.26,0.14,0.05,-0.01,7.3899,44537.78,44537.78,8470.00,1404276.34,1404276.34
3,688001.SH,2025-09-25,13.68,-9.92,0.32,0.20,0.06,-0.02,-0.03,0.07,...,0.28,0.16,0.07,0.00,7.5188,44537.78,44537.78,8470.00,1428772.12,1428772.12
4,688001.SH,2025-09-24,14.34,-9.26,0.35,0.22,0.08,0.00,-0.01,0.09,...,0.31,0.19,0.09,0.02,7.6735,44537.78,44537.78,8470.00,1458167.06,1458167.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588471,688981.SH,2020-07-22,9.57,-15.23,0.10,0.08,0.00,-0.02,-0.08,0.01,...,0.01,0.01,0.00,0.00,24.7978,741547.53,104023.12,104023.12,59004937.30,8277119.84
588472,688981.SH,2020-07-21,8.63,-16.17,0.09,0.07,0.00,-0.04,-0.11,0.00,...,0.01,0.00,-0.01,-0.01,24.5048,741547.53,104023.12,104023.12,58307882.62,8179338.11
588473,688981.SH,2020-07-20,9.17,-15.63,0.11,0.08,0.04,-0.06,-0.11,0.02,...,0.03,0.00,0.00,-0.01,24.6731,741547.53,104023.12,104023.12,58708318.29,8235510.59
588474,688981.SH,2020-07-17,2.26,-17.74,0.01,-0.01,-0.05,-0.13,-0.16,-0.07,...,-0.02,-0.04,-0.05,-0.06,23.1118,713642.32,104023.12,104023.12,54993277.38,8016021.80


In [1]:
label_dist = print_label_distribution(market_df["top_or_bottom"])
minority_class_counts = label_dist.min()
majority_class_counts = label_dist.max()

NameError: name 'print_label_distribution' is not defined

In [7]:
# Check for missing values in market_df_train
na_counts = market_df.isna().sum()
print("Columns with missing values:")
print(na_counts[na_counts > 0].sort_values(ascending=False))

Columns with missing values:
pe_ttm                 119637
vol_status_ma200       117052
close_ma200_diff       116473
low_ma200_diff         116473
high_ma200_diff        116473
                        ...  
low_cost_50pct_diff         1
low_cost_85pct_diff         1
low_cost_95pct_diff         1
winner_rate                 1
circ_mv                     1
Length: 282, dtype: int64


In [8]:
market_df = market_df.fillna(market_df.median(numeric_only=True))

In [46]:
market_df_copy = market_df.copy()

In [12]:
del market_df

In [7]:
# Drop columns with more than 100000 missing values
market_df = market_df.dropna(axis=1, thresh=market_df.shape[0] - 50)

In [9]:
# Check for missing values in market_df_train
na_counts = market_df.isna().sum()
print("Columns with missing values:")
print(na_counts[na_counts > 0].sort_values(ascending=False))

Columns with missing values:
Series([], dtype: int64)


In [9]:
market_df = market_df.dropna()

In [10]:
market_df

,ts_code,trade_date,close_his_low_diff,close_his_high_diff,close_cost_5pct_diff,close_cost_15pct_diff,close_cost_50pct_diff,close_cost_85pct_diff,close_cost_95pct_diff,close_weight_avg_diff,...,ps_200d_25pct_diff,ps_200d_50pct_diff,ps_200d_75pct_diff,ps_200d_90pct_diff,ps_ttm,total_share,float_share,free_share,total_mv,circ_mv
0,688001.SH,2025-09-30,13.19,-10.41,0.29,0.16,0.03,-0.03,-0.04,0.05,...,0.27,0.15,0.05,-0.01,7.4040,44537.78,44537.78,8470.00,1406948.61,1406948.61
1,688001.SH,2025-09-29,13.40,-10.20,0.30,0.17,0.05,-0.02,-0.04,0.06,...,0.27,0.15,0.06,0.00,7.4532,44537.78,44537.78,8470.00,1416301.54,1416301.54
2,688001.SH,2025-09-26,13.13,-10.47,0.29,0.17,0.04,-0.03,-0.04,0.05,...,0.26,0.14,0.05,-0.01,7.3899,44537.78,44537.78,8470.00,1404276.34,1404276.34
3,688001.SH,2025-09-25,13.68,-9.92,0.32,0.20,0.06,-0.02,-0.03,0.07,...,0.28,0.16,0.07,0.00,7.5188,44537.78,44537.78,8470.00,1428772.12,1428772.12
4,688001.SH,2025-09-24,14.34,-9.26,0.35,0.22,0.08,0.00,-0.01,0.09,...,0.31,0.19,0.09,0.02,7.6735,44537.78,44537.78,8470.00,1458167.06,1458167.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588471,688981.SH,2020-07-22,9.57,-15.23,0.10,0.08,0.00,-0.02,-0.08,0.01,...,0.01,0.01,0.00,0.00,24.7978,741547.53,104023.12,104023.12,59004937.30,8277119.84
588472,688981.SH,2020-07-21,8.63,-16.17,0.09,0.07,0.00,-0.04,-0.11,0.00,...,0.01,0.00,-0.01,-0.01,24.5048,741547.53,104023.12,104023.12,58307882.62,8179338.11
588473,688981.SH,2020-07-20,9.17,-15.63,0.11,0.08,0.04,-0.06,-0.11,0.02,...,0.03,0.00,0.00,-0.01,24.6731,741547.53,104023.12,104023.12,58708318.29,8235510.59
588474,688981.SH,2020-07-17,2.26,-17.74,0.01,-0.01,-0.05,-0.13,-0.16,-0.07,...,-0.02,-0.04,-0.05,-0.06,23.1118,713642.32,104023.12,104023.12,54993277.38,8016021.80


In [11]:
X = market_df.drop(columns=['trade_date', 'ts_code'] + labels)
y = market_df[labels]

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset based on time column 'trade_date'
# Assume 'trade_date' is available as a column in X or y (if not, you need to add it back)
# Here, we use the last 20% of dates as the test set

# Split the dataset by both ts_code and trade_date to ensure no leakage between stocks

# First, get all unique ts_codes
def time_series_split_by_stock(market_df, X, y, time_col='trade_date', stock_col='ts_code', split_ratio=0.8):
    """
    Split X and y into train/test sets by time for each stock, using the last (1-split_ratio) as test.

    Args:
        market_df (pd.DataFrame): DataFrame containing at least stock_col and time_col.
        X (pd.DataFrame): Features DataFrame (index must align with market_df).
        y (pd.DataFrame): Labels DataFrame (index must align with market_df).
        time_col (str): Name of the time column.
        stock_col (str): Name of the stock code column.
        split_ratio (float): Fraction of samples per stock to use for training.

    Returns:
        X_train, X_test, y_train, y_test (all pd.DataFrame)
    """
    unique_ts_codes = market_df[stock_col].unique()
    train_indices = []
    test_indices = []

    for ts in unique_ts_codes:
        ts_df = market_df[market_df[stock_col] == ts]
        ts_sorted = ts_df.sort_values(time_col)
        n = len(ts_sorted)
        split_idx = int(n * split_ratio)
        train_indices.extend(ts_sorted.index[:split_idx])
        test_indices.extend(ts_sorted.index[split_idx:])

    X_train = X.loc[train_indices]
    X_test = X.loc[test_indices]
    y_train = y.loc[train_indices]
    y_test = y.loc[test_indices]

    # Check alignment
    assert all(X_train.index == y_train.index)
    assert all(X_test.index == y_test.index)
    return X_train, X_test, y_train, y_test

In [13]:
# Check consistency of X_train, X_test, y_train, y_test

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# Check index alignment
print("X_train and y_train index equal:", all(X_train.index == y_train.index))
print("X_test and y_test index equal:", all(X_test.index == y_test.index))

# Check for missing values
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_test:", y_test.isnull().sum().sum())

X_train shape: (470533, 383)
y_train shape: (470533, 6)
X_test shape: (117943, 383)
y_test shape: (117943, 6)
X_train and y_train index equal: True
X_test and y_test index equal: True
Missing values in X_train: 0
Missing values in y_train: 0
Missing values in X_test: 0
Missing values in y_test: 0


In [23]:
X_train_copy = X_train.copy()
y_train_copy = y_train.copy()

In [20]:
X_train = X_train_copy.copy()
y_train = y_train_copy.copy()

In [14]:
train_label_dist = print_label_distribution(y_train["top_or_bottom"])
minority_class_counts = train_label_dist.min()
majority_class_counts = train_label_dist.max()

Current label distribution:
top_or_bottom
0    434584
2     18009
1     17940
Name: count, dtype: int64


In [ ]:
def smote_resample(X, y, minority_count, factor=10):
    """
    Apply SMOTE oversampling to balance classes.

    Args:
        X (pd.DataFrame): Features.
        y (pd.Series): Target labels.
        minority_count (int): Count of minority class.
        factor (int): Oversampling factor for minority classes.

    Returns:
        X_resampled, y_resampled: Resampled features and labels.
    """
    from imblearn.over_sampling import SMOTE
    sampling_strategy = {1: int(minority_count * factor), 2: int(minority_count * factor)}
    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    return X_resampled, y_resampled

# Example usage:
# X_train_resample, y_train_resample = smote_resample(X_train, y_train["top_or_bottom"], minority_class_counts, factor=10)

In [16]:
from imblearn.under_sampling import RandomUnderSampler

# Calculate the minority class count
minority_count = minority_class_counts

# Set majority class to be at most the minority class count
sampling_strategy = {cls: min(count, minority_count) for cls, count in y_train["top_or_bottom"].value_counts().items()}

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_train_resample, y_train_resample = rus.fit_resample(X_train, y_train["top_or_bottom"])

print_label_distribution(y_train_resample)

Current label distribution:
top_or_bottom
0    17940
1    17940
2    17940
Name: count, dtype: int64


top_or_bottom
0    17940
1    17940
2    17940
Name: count, dtype: int64

help to generate the classification train codes by different methods in XGB, RF, LGBM, CatBoost, DeepLearning. print the test results. save the test results of each methods into tthe plain text file




In [30]:
print_label_distribution(y_train_resample)

Current label distribution:
top_or_bottom
0    434584
1    179400
2    179400
Name: count, dtype: int64


top_or_bottom
0    434584
1    179400
2    179400
Name: count, dtype: int64

In [17]:
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import numpy as np
from datetime import datetime
from sklearn.utils.class_weight import compute_sample_weight
import joblib


In [18]:
train_comments = "Majority class has undersampling applied"

In [19]:
def save_results_to_txt(results, model_name, comments=None, surfix=None):
    """
    Save classification results to a plain text file.

    Args:
        results (dict): Dictionary of model results.
        model_name (str): Model name to use in the filename.
        comments (str, optional): Training comments to include.
        surfix (str, optional): CSV/model surfix for filename.
    """
    if surfix is None:
        surfix = csv_surfix  # use global if not provided
    if comments is None:
        comments = train_comments  # use global if not provided

    file_path = f"static/bundle/{surfix}_classification_results_{model_name}.txt"
    with open(file_path, "a") as f:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"=== Training Timestamp: {timestamp} ===\n")
        f.write(f"=== Training Comments : {comments} ===\n")
        for k, v in results.items():
            f.write(f"=== {k} ===\n{v}\n\n")
    print(f"Results saved to {file_path}")

# Example usage:
# save_results_to_txt(results, model_name="cost")

In [20]:
import joblib

def save_sklearn_model(model, model_name, prefix, acc):
    """
    Save a sklearn model to disk with a filename containing prefix, model name, and accuracy.

    Args:
        model: Trained sklearn model object.
        model_name (str): Name of the model (e.g., 'RF', 'XGB').
        prefix (str): CSV/model prefix (e.g., '688').
        acc (float): Model accuracy (will be formatted to 4 decimals).
    """
    model_path = f"static/bundle/{prefix}_{model_name}_model_{acc:.4f}.pkl"
    joblib.dump(model, model_path)
    print(f"{model_name} model saved to {model_path}")

# Example usage:
# save_sklearn_model(rf_clf, "RF", csv_surfix, rf_acc)

In [31]:
results = {}

label = labels[0]

print(f"\n=== Classification for label: {label} ===")
y_train_label = y_train_resample
y_test_label = y_test[label]

# Compute balanced sample weights for training
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_label)



=== Classification for label: top_or_bottom ===


In [32]:
# Deep Learning (Keras)
num_classes = len(np.unique(y_train_label))
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train_label, epochs=30, batch_size=32, verbose=0)
dl_pred = np.argmax(model.predict(X_test), axis=1)
dl_acc = accuracy_score(y_test_label, dl_pred)
dl_report = classification_report(y_test_label, dl_pred)
print(f"\n[DeepLearning] Accuracy: {dl_acc:.4f}\n{dl_report}")
results[f"{label}_DeepLearning"] = f"Accuracy: {dl_acc:.4f}\n{dl_report}"

c:\Users\HANJ29\Development\vdev1\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7356/7356 ━━━━━━━━━━━━━━━━━━━━ 5s 680us/step

[DeepLearning] Accuracy: 0.9234
              precision    recall  f1-score   support

           0       0.92      1.00      0.96    217368
           1       0.00      0.00      0.00      8692
           2       0.00      0.00      0.00      9329

    accuracy                           0.92    235389
   macro avg       0.31      0.33      0.32    235389
weighted avg       0.85      0.92      0.89    235389



c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [22]:
from utils import train_utils
import importlib
importlib.reload(train_utils)

<module 'utils.train_utils' from 'c:\\Users\\HANJ29\\Development\\code\\sms\\smartinvestor_ln\\training\\utils\\train_utils.py'>

In [23]:
# Print feature importance from Random Forest
def get_top_features_by_importance(rf_clf, feature_names, threshold=0.9, verbose=True):
    """
    Get top features whose cumulative importance sum exceeds the given threshold.

    Args:
        rf_clf: Trained RandomForestClassifier.
        feature_names: List or Index of feature names.
        threshold: Cumulative importance threshold (default 0.9).
        verbose: If True, print the selected features and their importances.

    Returns:
        List of top feature names.
    """
    importances = rf_clf.feature_importances_
    sorted_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)
    top_features = []
    cum_sum = 0.0
    for feat, imp in sorted_features:
        top_features.append(feat)
        cum_sum += imp
        if verbose:
            print(f"{feat}: {imp:.4f} (cumulative: {cum_sum:.4f})")
        if cum_sum >= threshold:
            break
    if verbose:
        print(f"\nSelected {len(top_features)} features with cumulative importance >= {threshold}")
    return top_features

# Example usage:


In [71]:
top_features = get_top_features_by_importance(rf_clf, X_train.columns, threshold=0.9)


close_his_high_diff: 0.2720 (cumulative: 0.2720)
close_his_low_diff: 0.2550 (cumulative: 0.5270)
close_cost_15pct_diff: 0.0472 (cumulative: 0.5742)
close_cost_5pct_diff: 0.0460 (cumulative: 0.6202)
close_cost_50pct_diff: 0.0299 (cumulative: 0.6501)
close_cost_85pct_diff: 0.0258 (cumulative: 0.6759)
low_weight_avg_diff: 0.0252 (cumulative: 0.7012)
close_cost_95pct_diff: 0.0221 (cumulative: 0.7233)
open_pre_close_pct_chg: 0.0216 (cumulative: 0.7448)
high_cost_15pct_diff: 0.0165 (cumulative: 0.7613)
high_cost_5pct_diff: 0.0153 (cumulative: 0.7766)
close_weight_avg_diff: 0.0143 (cumulative: 0.7909)
high_weight_avg_diff: 0.0135 (cumulative: 0.8045)
vol_120d_25pct_diff: 0.0129 (cumulative: 0.8174)
low_cost_15pct_diff: 0.0104 (cumulative: 0.8278)
high_cost_50pct_diff: 0.0092 (cumulative: 0.8370)
low_cost_5pct_diff: 0.0082 (cumulative: 0.8451)
low_cost_50pct_diff: 0.0073 (cumulative: 0.8524)
high_cost_85pct_diff: 0.0066 (cumulative: 0.8590)
vol_30d_10pct_diff: 0.0062 (cumulative: 0.8652)
high_

In [32]:
# Random Forest
from sklearn.metrics import f1_score
from collections import Counter


params_rf["max_depth"] = 150
params_rf["n_estimators"] = 100  # more trees for stability
params_rf["min_samples_split"] = 2
params_rf["min_samples_leaf"] = 1
# params_rf["class_weight"] = class_weight  # custom weights
# params_rf["max_features"] = "sqrt"

# Optionally, set bootstrap=False for more variance (uncomment if needed)
# params_rf["bootstrap"] = False
# Optionally, you can tune further with grid search or set bootstrap=False for more variance.
train_comments = f"Used top features based on importance, max_depth={params_rf['max_depth']}, n_estimators={params_rf['n_estimators']}, resampled data to balance classes. minority class has been oversampled to 25 times."

rf_clf = RandomForestClassifier(**params_rf)
rf_clf.fit(X_train_resample, y_train_label, sample_weight=sample_weight)
rf_pred = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test_label, rf_pred)
rf_report = classification_report(y_test_label, rf_pred)

rf_f1_macro = f1_score(y_test_label, rf_pred, average="macro")
print(f"[RandomForest] Macro F1: {rf_f1_macro:.4f}")
print(f"\n[RandomForest] Accuracy: {rf_acc:.4f}\n{rf_report}")
results[f"{label}_RF"] = (
    f"Accuracy: {rf_acc:.4f}\n{rf_report}\nMacro F1: {rf_f1_macro:.4f}"
)

save_results_to_txt(results, model_name="RF")

[RandomForest] Macro F1: 0.4023

[RandomForest] Accuracy: 0.9242
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    108878
           1       0.68      0.03      0.05      4458
           2       0.53      0.12      0.20      4607

    accuracy                           0.92    117943
   macro avg       0.71      0.38      0.40    117943
weighted avg       0.90      0.92      0.90    117943

Results saved to static/bundle/688_classification_results_RF.txt


In [19]:
save_sklearn_model(rf_clf, "RF", csv_surfix, rf_acc)

RF model saved to static/bundle/3_RF_model_0.9855.pkl


In [32]:
import joblib
# Load the saved Random Forest model
rf_loaded = joblib.load("static/bundle/688_RF_model_0.9849.pkl")

# Print the input features list used by the model
print("Input features used by the loaded RF model:")
print(rf_loaded.feature_names_in_)

Input features used by the loaded RF model:
['open_pre_close_change' 'open_pre_close_pct_chg' 'change' 'pct_change'
 'pct_vol_chg' 'vol_status_ma6' 'vol_status_ma10' 'vol_status_ma16'
 'vol_status_ma25' 'vol_status_ma43' 'vol_status_ma60' 'vol_status_ma90'
 'vol_status_ma120' 'vol_status_ma200' 'vol_30d_10pct_diff'
 'vol_30d_25pct_diff' 'vol_30d_50pct_diff' 'vol_30d_75pct_diff'
 'vol_30d_90pct_diff' 'vol_60d_10pct_diff' 'vol_60d_25pct_diff'
 'vol_60d_50pct_diff' 'vol_60d_75pct_diff' 'vol_60d_90pct_diff'
 'vol_90d_10pct_diff' 'vol_90d_25pct_diff' 'vol_90d_50pct_diff'
 'vol_90d_75pct_diff' 'vol_90d_90pct_diff' 'vol_120d_10pct_diff'
 'vol_120d_25pct_diff' 'vol_120d_50pct_diff' 'vol_120d_75pct_diff'
 'vol_120d_90pct_diff' 'vol_200d_10pct_diff' 'vol_200d_25pct_diff'
 'vol_200d_50pct_diff' 'vol_200d_75pct_diff' 'vol_200d_90pct_diff'
 'amount_30d_10pct_diff' 'amount_30d_25pct_diff' 'amount_30d_50pct_diff'
 'amount_30d_75pct_diff' 'amount_30d_90pct_diff' 'amount_60d_10pct_diff'
 'amount_60d_25

In [ ]:

# XGBoost
xgb_clf = xgb.XGBClassifier(**params_xgb)
xgb_clf.fit(X_train_resample, y_train_label, sample_weight=sample_weight)
xgb_pred = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test_label, xgb_pred)
xgb_report = classification_report(y_test_label, xgb_pred)
print(f"\n[XGBoost] Accuracy: {xgb_acc:.4f}\n{xgb_report}")
results[f"{label}_XGB"] = f"Accuracy: {xgb_acc:.4f}\n{xgb_report}"


In [ ]:

# LightGBM
lgb_clf = lgb.LGBMClassifier(**params_lgb)
lgb_clf.fit(X_train_resample, y_train_label, sample_weight=sample_weight)
lgb_pred = lgb_clf.predict(X_test)
lgb_acc = accuracy_score(y_test_label, lgb_pred)
lgb_report = classification_report(y_test_label, lgb_pred)
print(f"\n[LightGBM] Accuracy: {lgb_acc:.4f}\n{lgb_report}")
results[f"{label}_LGBM"] = f"Accuracy: {lgb_acc:.4f}\n{lgb_report}"


In [ ]:

# CatBoost
cat_clf = CatBoostClassifier(**params_cat)
cat_clf.fit(X_train_resample, y_train_label, sample_weight=sample_weight, verbose=False)
cat_pred = cat_clf.predict(X_test)
cat_acc = accuracy_score(y_test_label, cat_pred)
cat_report = classification_report(y_test_label, cat_pred)
print(f"\n[CatBoost] Accuracy: {cat_acc:.4f}\n{cat_report}")
results[f"{label}_CatBoost"] = f"Accuracy: {cat_acc:.4f}\n{cat_report}"



In [27]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 1) 标准化特征（用训练集拟合，应用到测试集）
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train.values)             # 用未下采样/过采样的训练特征
X_test_np  = scaler.transform(X_test.values)

# 若你保留了适度过采样的 y_train_resample/X_train_resample，请改用对应变量：
# X_train_np = scaler.fit_transform(X_train_resample.values)
# y_train_label = y_train_resample

y_train_label = y_train["top_or_bottom"].values  # 推荐用原分布训练配合 class_weight
y_test_label  = y_test[label].values

num_classes = len(np.unique(y_train_label))
input_dim = X_train_np.shape[1]  # 400

# 2) 计算 class_weight 基于原始分布（越少的类权重越大）
classes = np.unique(y_train_label)
class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_label)
class_weight = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}

# 可选：Focal Loss（多分类），用 alpha 反映类权重
# tf >= 2.11 提供 CategoricalFocalCrossentropy；若使用 sparse 标签，需要 one-hot
use_focal = True
if use_focal:
    # 将类别权重归一化到 alpha（按比例）
    alpha = class_weights_arr / class_weights_arr.sum()
    loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(alpha=alpha, gamma=2.0)
    # 将标签转换为 one-hot
    y_train_oh = tf.keras.utils.to_categorical(y_train_label, num_classes=num_classes)
    y_test_oh  = tf.keras.utils.to_categorical(y_test_label,  num_classes=num_classes)
else:
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
    y_train_oh = None
    y_test_oh  = None

# 3) 模型结构（BatchNorm + Dropout + L2）
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,), kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=loss_fn,
              metrics=['accuracy',
                       tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall')])

# 4) 回调：早停 + 学习率调度
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=5e-5)
]

# 5) 训练（使用验证集，避免过拟合）
if use_focal:
    history = model.fit(X_train_np, y_train_oh,
                        epochs=50, batch_size=512, verbose=1,
                        validation_split=0.1,
                        callbacks=callbacks)
else:
    history = model.fit(X_train_np, y_train_label,
                        epochs=50, batch_size=512, verbose=1,
                        validation_split=0.1,
                        callbacks=callbacks,
                        class_weight=class_weight)

# 6) 预测与后验调整（可选）：按训练先验对 softmax 概率再加权，减小少数类过预测
probs = model.predict(X_test_np)
# 训练先验（原分布）
prior = np.array([ (y_train_label == c).mean() for c in classes ])
prior = prior / prior.sum()
# 将概率除以先验，并重新归一化（等价于校正到更接近均匀先验）
adjusted = probs / (prior + 1e-8)
adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)

# 7) 最终决策可以用 adjusted 的 argmax
dl_pred = adjusted.argmax(axis=1)
dl_acc = accuracy_score(y_test_label, dl_pred)
dl_report = classification_report(y_test_label, dl_pred)
print(f"\n[DeepLearning] Accuracy: {dl_acc:.4f}\n{dl_report}")
results[f"{label}_DeepLearning"] = f"Accuracy: {dl_acc:.4f}\n{dl_report}"

# Save results to plain text file
with open(f"static/bundle/{csv_surfix}_classification_results_DL.txt", "a") as f:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    f.write(f"=== Training Timestamp: {timestamp} ===\n")
    f.write(f"=== Training Comments : {train_comments} ===\n")
    for k, v in results.items():
        f.write(f"=== {k} ===\n{v}\n\n")

c:\Users\HANJ29\Development\vdev1\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
1656/1656 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.4022 - loss: 0.0265 - precision: 0.2151 - recall: 0.1043 - val_accuracy: 0.3434 - val_loss: 0.0190 - val_precision: 0.1139 - val_recall: 0.0497 - learning_rate: 0.0010
Epoch 2/50
1656/1656 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.3834 - loss: 0.0178 - precision: 0.1353 - recall: 0.0603 - val_accuracy: 0.4051 - val_loss: 0.0169 - val_precision: 0.1162 - val_recall: 0.0482 - learning_rate: 0.0010
Epoch 3/50
1656/1656 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.3765 - loss: 0.0168 - precision: 0.1319 - recall: 0.0587 - val_accuracy: 0.4639 - val_loss: 0.0166 - val_precision: 0.1243 - val_recall: 0.0466 - learning_rate: 0.0010
Epoch 4/50
1656/1656 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.3799 - loss: 0.0167 - precision: 0.1279 - recall: 0.0564 - val_accuracy: 0.3761 - val_loss: 0.0163 - val_precision: 0.1183 - val_recall: 0.0520 - learning_rate: 0.0010
Epoch 5/50
1656/1656 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step -

c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HANJ29\Development\vdev1\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [2]:
# 加载模型后查看参数
import joblib
rf_clf = joblib.load('static/bundle/3_RF_model_0.9855.pkl')
print(rf_clf.get_params())
print("Features used by the loaded RF model:")
print(rf_clf.feature_names_in_)

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Features used by the loaded RF model:
['open_pre_close_change' 'open_pre_close_pct_chg' 'change' 'pct_change'
 'pct_vol_chg' 'vol_status_ma6' 'vol_status_ma10' 'vol_status_ma16'
 'vol_status_ma25' 'vol_status_ma43' 'vol_status_ma60' 'vol_status_ma90'
 'vol_status_ma120' 'vol_status_ma200' 'vol_30d_10pct_diff'
 'vol_30d_25pct_diff' 'vol_30d_50pct_diff' 'vol_30d_75pct_diff'
 'vol_30d_90pct_diff' 'vol_60d_10pct_diff' 'vol_60d_25pct_diff'
 'vol_60d_50pct_diff' 'vol_60d_75pct_diff' 'vol_60d_90pct_diff'
 'vol_90d_10pct_diff' 'vol_90d_25pct_diff' 'vol_90d_50pct_diff'
 'vol_90d_75pct_diff'